**Навигация** : [Индекс](../../README.md) | [<< Назад](01-1-OpenAI-DALL-E-3.ipynb) | [Далее >>](01-3-Basic-Image-Operations.ipynb)

# 🤖 GPT-5 Мультимодальный - Анализ и генерация изображений

**Модуль :** 01-Images-Foundation  
**Уровень :** 🟢 Начинающий  
**Технологии :** GPT-5, OpenRouter API, Vision AI  
**Оценочная длительность :** 30 минут  

## 🎯 Цели обучения

- [ ] Настроить GPT-5 через OpenRouter для анализа изображений
- [ ] Освоить мультимодальные диалоги текст + изображение
- [ ] Анализировать и описывать изображения с точностью
- [ ] Создавать оптимизированные промпты для визуального анализа
- [ ] Интегрировать GPT-5 в педагогические сценарии использования

## 📚 Предварительные требования

- Environment Setup (модуль 00) завершён
- Ключ API OpenRouter настроен
- Базовые знания в области мультимодального ИИ

## ⚡ Возможности GPT-5

- **Продвинутое зрение** : Детальный анализ изображений
- **Мультимодальность** : Одновременный диалог текст + изображение
- **Расширенный контекст** : До 400K токенов [OpenAI 2025]
- **Рассуждение** : Сложный и дедуктивный анализ
- **Образование** : Идеально для обучения

In [1]:
# Parametres Papermill - JAMAIS modifier ce commentaire

# Configuration conversation
notebook_mode = "interactive"        # "interactive" ou "batch"
skip_widgets = False               # True pour mode batch MCP
debug_level = "INFO"               

# Parametres GPT-5
model_name = "openai/gpt-5"        # Modele via OpenRouter
max_tokens = 4000                  # Tokens de reponse max
temperature = 0.7                  # Creativite (0.0-1.0)
top_p = 0.9                        # Diversite sampling

# Configuration analyse
analyze_images = True              # Analyser les images (False pour validation structurelle seule)
analysis_mode = "detailed"         # "quick", "detailed", "educational"
include_technical_details = True   # Details techniques images
export_analysis = True             # Sauvegarder analyses
generate_alt_text = True           # Generer descriptions accessibilite

# Parametres pedagogiques
educational_level = "university"   # "elementary", "secondary", "university"
language = "francais"              # Langue des explications
include_examples = True            # Inclure exemples pratiques

In [2]:
# Parameters
BATCH_MODE = "true"


## Настройка окружения

Загрузим зависимости, настроим API OpenRouter для доступа к GPT-5 и подготовим вспомогательные функции для обработки изображений.

In [3]:
# Verification des dependances externes
import importlib

_DEPS_STATUS = {}
try:
    importlib.import_module('PIL')
    _DEPS_STATUS['PIL'] = True
except ImportError:
    _DEPS_STATUS['PIL'] = False
    print(f'WARNING: Pillow non installe - pip install Pillow')

try:
    importlib.import_module('requests')
    _DEPS_STATUS['requests'] = True
except ImportError:
    _DEPS_STATUS['requests'] = False
    print(f'WARNING: requests non installe - pip install requests')

try:
    importlib.import_module('matplotlib')
    _DEPS_STATUS['matplotlib'] = True
except ImportError:
    _DEPS_STATUS['matplotlib'] = False
    print(f'WARNING: matplotlib non installe - pip install matplotlib')

try:
    importlib.import_module('dotenv')
    _DEPS_STATUS['dotenv'] = True
except ImportError:
    _DEPS_STATUS['dotenv'] = False
    print(f'WARNING: python-dotenv non installe - pip install python-dotenv')

_all_deps_ok = all(_DEPS_STATUS.values())
if not _all_deps_ok:
    missing = [k for k, v in _DEPS_STATUS.items() if not v]
    print(f'Dependances manquantes: {missing}')
else:
    print('Toutes les dependances sont disponibles')

# Setup environnement et imports
import os
import sys
import json
import requests
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Any, Optional, Union
import base64
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
import logging
from urllib.parse import urlparse

# Chargement robuste de la configuration .env
from dotenv import load_dotenv

current_path = Path.cwd()
env_loaded = False
for _ in range(10):
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path.name}")
        env_loaded = True
        break
    if current_path.name == "GenAI" or len(current_path.parts) <= 1:
        break
    current_path = current_path.parent
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")

# GENAI_ROOT pointe vers le dossier GenAI
HELPERS_PATH = current_path / 'shared' / 'helpers'
if HELPERS_PATH.exists():
    sys.path.insert(0, str(HELPERS_PATH.parent))
    try:
        from helpers.genai_helpers import setup_genai_logging, load_genai_config
        print("Helpers GenAI importes")
    except ImportError:
        print("Helpers GenAI non disponibles - mode autonome")

# Configuration logging
logging.basicConfig(level=getattr(logging, debug_level))
logger = logging.getLogger('gpt5_multimodal')

print(f"GPT-5 Multimodal - Analyse et Generation d'Images")
print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Mode: {notebook_mode}, Analyse: {analysis_mode}, Niveau: {educational_level}")
print(f"Langue: {language}, Max tokens: {max_tokens}")

Toutes les dependances sont disponibles


.env charge depuis: .env
Helpers GenAI importes
GPT-5 Multimodal - Analyse et Generation d'Images
2026-06-24 06:44:53
Mode: interactive, Analyse: detailed, Niveau: university
Langue: francais, Max tokens: 4000


Окружение инициализировано. Следующая ячейка настраивает доступ к API OpenRouter и проверяет, что модель GPT-5 доступна для мультимодальных запросов.

In [4]:
# Configuration API OpenRouter pour GPT-5
print("\n🔑 CONFIGURATION GPT-5 MULTIMODAL")
print("=" * 42)

# Vérification clé API
openrouter_key = os.getenv('OPENROUTER_API_KEY') or os.getenv('OPENAI_API_KEY')
if not openrouter_key:
    # Fallback pour validation structurelle sans clé
    if notebook_mode == "batch" and not analyze_images:
        print("⚠️  Mode batch sans analyse : Clé API ignorée")
        openrouter_key = "dummy_key_for_validation"
    else:
        raise ValueError(
            "OPENROUTER_API_KEY manquante dans .env\n"
            "Configurez votre clé dans le fichier .env (voir .env.example)"
        )

print(f"✅ Clé API OpenRouter configurée")

# Configuration headers et endpoint
api_base_url = os.getenv("OPENROUTER_BASE_URL", os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1"))
headers = {
    "Authorization": f"Bearer {openrouter_key}",
    "HTTP-Referer": "https://coursia.myia.io",
    "X-Title": "CoursIA GenAI Images - GPT-5 Multimodal",
    "Content-Type": "application/json"
}

# Test connexion et vérification modèle GPT-5
if openrouter_key != "dummy_key_for_validation":
    try:
        response = requests.get(f"{api_base_url}/models", headers=headers, timeout=10)
        if response.status_code == 200:
            models_data = response.json()
            gpt5_models = [m for m in models_data.get('data', []) if 'gpt-5' in m.get('id', '').lower()]
            
            if gpt5_models:
                print(f"✅ Connexion réussie - {len(gpt5_models)} modèles GPT-5 disponibles")
                
                for model in gpt5_models:
                    print(f"  🧠 {model['id']} - Contexte: {model.get('context_length', 'N/A')} tokens")
                    if 'vision' in model.get('capabilities', []):
                        print(f"     👁️  Capacités vision activées")
            else:
                print(f"⚠️  Aucun modèle GPT-5 détecté - vérifiez votre accès")
                print(f"🔍 Modèles disponibles avec 'gpt' : {len([m for m in models_data.get('data', []) if 'gpt' in m.get('id', '').lower()])}")
        else:
            print(f"⚠️  Connexion API: HTTP {response.status_code}")
    except Exception as e:
        print(f"❌ Erreur connexion: {str(e)[:100]}...")
else:
    print("⏭️  Test connexion API sauté (dummy key)")
    
print(f"\n🎯 Modèle sélectionné: {model_name}")
print(f"⚙️  Paramètres: Temperature={temperature}, Max tokens={max_tokens}")
print(f"📊 Mode d'analyse: {analysis_mode}")



🔑 CONFIGURATION GPT-5 MULTIMODAL
✅ Clé API OpenRouter configurée


✅ Connexion réussie - 26 modèles GPT-5 disponibles
  🧠 openai/gpt-5.5-pro - Contexte: 1050000 tokens
  🧠 openai/gpt-5.5 - Contexte: 1050000 tokens
  🧠 openai/gpt-5.4-image-2 - Contexte: 272000 tokens
  🧠 openai/gpt-5.4-nano - Contexte: 400000 tokens
  🧠 openai/gpt-5.4-mini - Contexte: 400000 tokens
  🧠 openai/gpt-5.4-pro - Contexte: 1050000 tokens
  🧠 openai/gpt-5.4 - Contexte: 1050000 tokens
  🧠 openai/gpt-5.3-chat - Contexte: 128000 tokens
  🧠 openai/gpt-5.3-codex - Contexte: 400000 tokens
  🧠 openai/gpt-5.2-codex - Contexte: 400000 tokens
  🧠 openai/gpt-5.2-chat - Contexte: 128000 tokens
  🧠 openai/gpt-5.2-pro - Contexte: 400000 tokens
  🧠 openai/gpt-5.2 - Contexte: 400000 tokens
  🧠 openai/gpt-5.1-codex-max - Contexte: 400000 tokens
  🧠 openai/gpt-5.1 - Contexte: 400000 tokens
  🧠 openai/gpt-5.1-chat - Contexte: 128000 tokens
  🧠 openai/gpt-5.1-codex - Contexte: 400000 tokens
  🧠 openai/gpt-5.1-codex-mini - Contexte: 400000 tokens
  🧠 openai/gpt-5-image-mini - Contexte: 400000 toke

API настроен, и соединение проверено. Следующая ячейка определяет функции мультимодального анализа, которые инкапсулируют вызовы GPT-5 с кодированием изображений в base64.

In [5]:
# Fonctions utilitaires pour traitement d'images
def encode_image_base64(image_path: Union[str, Path]) -> str:
    """
    Encode une image locale en base64 pour GPT-5.
    
    Args:
        image_path: Chemin vers l'image locale
        
    Returns:
        String base64 de l'image
    """
    try:
        with open(image_path, "rb") as image_file:
            return base64.b64encode(image_file.read()).decode('utf-8')
    except Exception as e:
        raise ValueError(f"Erreur lecture image {image_path}: {str(e)}")

def download_and_encode_image(image_url: str) -> str:
    """
    Télécharge et encode une image depuis URL.
    
    Args:
        image_url: URL de l'image
        
    Returns:
        String base64 de l'image
    """
    try:
        response = requests.get(image_url, timeout=30)
        response.raise_for_status()
        return base64.b64encode(response.content).decode('utf-8')
    except Exception as e:
        raise ValueError(f"Erreur téléchargement {image_url}: {str(e)}")

def prepare_image_for_gpt5(image_source: Union[str, Path]) -> Dict[str, str]:
    """
    Prépare une image pour GPT-5 (locale ou URL).
    
    Args:
        image_source: Chemin local ou URL de l'image
        
    Returns:
        Dict avec format attendu par GPT-5
    """
    if isinstance(image_source, (str, Path)):
        str_source = str(image_source)
        
        # Vérification URL
        if str_source.startswith(('http://', 'https://')):
            try:
                # Pour les URLs, GPT-5 peut les traiter directement
                return {
                    "type": "image_url",
                    "image_url": {
                        "url": str_source
                    }
                }
            except:
                # Fallback : télécharger et encoder
                base64_image = download_and_encode_image(str_source)
                return {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
        else:
            # Image locale
            if Path(str_source).exists():
                base64_image = encode_image_base64(str_source)
                return {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{base64_image}"
                    }
                }
            else:
                raise FileNotFoundError(f"Image non trouvée: {str_source}")
    
    raise ValueError(f"Format d'image non supporté: {type(image_source)}")

print("✅ Fonctions utilitaires images prêtes")

✅ Fonctions utilitaires images prêtes


### Функция анализа и демонстрационные изображения

Следующая ячейка определяет основную функцию `analyze_image_with_gpt5()`, которая кодирует изображение в base64 и отправляет его в мультимодальный API. Затем мы подготавливаем набор тестовых изображений.

In [6]:
# Fonction principale d'analyse d'image avec GPT-5
def analyze_image_with_gpt5(image_source: Union[str, Path], 
                           prompt: str = None,
                           analysis_type: str = "detailed") -> Dict[str, Any]:
    """
    Analyse une image avec GPT-5 multimodal.
    
    Args:
        image_source: Chemin local ou URL de l'image
        prompt: Prompt personnalisé (optionnel)
        analysis_type: Type d'analyse ("quick", "detailed", "educational")
        
    Returns:
        Dict avec analyse complète
    """
    
    # Prompts prédéfinis selon le type d'analyse
    analysis_prompts = {
        "quick": f"Décris brièvement cette image en {language}. Sois concis mais précis.",
        
        "detailed": f"""Analyse cette image en détail en {language}. Inclus :
        
1. **Description générale** : Que voit-on dans l'image ?
2. **Éléments visuels** : Couleurs, composition, style artistique
3. **Contexte** : Époque, lieu, situation probable
4. **Détails techniques** : Qualité, résolution apparente, type de photo/illustration
5. **Émotions/Atmosphère** : Quelle ambiance dégage l'image ?
6. **Interprétation** : Signification possible, message artistique

Sois précis et pédagogique dans tes explications.""",
        
        "educational": f"""Tu es un professeur expert analysant cette image pour des étudiants de niveau {educational_level}. En {language}, fournis :

🎯 **ANALYSE PÉDAGOGIQUE**

**1. Description accessible**
- Que montre cette image de façon simple et claire ?

**2. Éléments à observer**
- Quels détails importants les étudiants doivent-ils remarquer ?
- Techniques artistiques ou photographiques utilisées

**3. Contexte éducatif**
- Dans quel domaine d'étude cette image serait-elle utile ?
- Quelles disciplines académiques peuvent l'utiliser ?

**4. Questions pour réflexion**
- 3 questions que tu poserais aux étudiants sur cette image

**5. Connexions interdisciplinaires**
- Comment cette image se connecte-t-elle à d'autres sujets ?

Adapte ton vocabulaire au niveau {educational_level}."""
    }
    
    # Sélection du prompt
    if prompt is None:
        prompt = analysis_prompts.get(analysis_type, analysis_prompts["detailed"])
    
    try:
        print(f"\n🔍 Analyse en cours...")
        print(f"📝 Type: {analysis_type}")
        print(f"🖼️  Source: {str(image_source)[:100]}{'...' if len(str(image_source)) > 100 else ''}")
        
        # Préparation de l'image
        image_data = prepare_image_for_gpt5(image_source)
        
        # Construction du message multimodal
        messages = [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": prompt
                    },
                    image_data
                ]
            }
        ]
        
        # Payload de la requête
        payload = {
            "model": model_name,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": temperature,
            "top_p": top_p
        }
        
        # Requête API
        start_time = datetime.now()
        response = requests.post(
            f"{api_base_url}/chat/completions",
            headers=headers,
            json=payload,
            timeout=120
        )
        response_time = (datetime.now() - start_time).total_seconds()
        
        if response.status_code == 200:
            result = response.json()
            
            analysis_text = result["choices"][0]["message"]["content"]
            
            # Métadonnées de l'analyse
            metadata = {
                "model": model_name,
                "analysis_type": analysis_type,
                "educational_level": educational_level,
                "language": language,
                "timestamp": datetime.now().isoformat(),
                "response_time": response_time,
                "tokens_used": result.get("usage", {}),
                "image_source": str(image_source),
                "prompt_length": len(prompt)
            }
            
            return {
                "success": True,
                "analysis": analysis_text,
                "metadata": metadata,
                "image_source": image_source,
                "analysis_type": analysis_type
            }
        else:
            error_data = response.json() if response.headers.get('content-type', '').startswith('application/json') else {}
            error_msg = error_data.get("error", {}).get("message", f"HTTP {response.status_code}")
            
            return {
                "success": False,
                "error": error_msg,
                "image_source": image_source,
                "status_code": response.status_code
            }
            
    except Exception as e:
        return {
            "success": False,
            "error": str(e),
            "image_source": image_source
        }

print("✅ Fonction d'analyse GPT-5 prête")

✅ Fonction d'analyse GPT-5 prête


Примеры изображений и педагогические промпты определены. Следующая ячейка запускает демонстрационный анализ, тестируя три режима (quick, detailed, educational) на конкретном изображении.

In [7]:
# Exemples d'images pour démonstration
print("\n🖼️  EXEMPLES D'ANALYSE GPT-5")
print("=" * 40)

# Images d'exemple (URLs publiques pour tests)
example_images = [
    {
        "title": "🏛️ Architecture Historique",
        "url": "https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800",  # Colisée Rome
        "description": "Monument historique romain - idéal pour analyse architecturale",
        "category": "Histoire/Architecture"
    },
    {
        "title": "🔬 Science et Technologie",
        "url": "https://images.unsplash.com/photo-1532094349884-543bc11b234d?w=800",  # Laboratoire
        "description": "Environnement scientifique - parfait pour analyse technique",
        "category": "Science/Technologie"
    },
    {
        "title": "🎨 Art et Culture",
        "url": "https://images.unsplash.com/photo-1541961017774-22349e4a1262?w=800",  # Peinture
        "description": "Œuvre artistique - excellent pour analyse esthétique",
        "category": "Art/Culture"
    },
    {
        "title": "🌍 Nature et Environnement",
        "url": "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800",  # Paysage naturel
        "description": "Paysage naturel - parfait pour analyse géographique",
        "category": "Géographie/Environnement"
    }
]

# Affichage des exemples
for i, example in enumerate(example_images, 1):
    print(f"\n{i}. {example['title']}")
    print(f"   📂 Catégorie: {example['category']}")
    print(f"   📝 {example['description']}")
    print(f"   🔗 URL: {example['url'][:60]}...")

print(f"\n💡 Conseils pour l'analyse avec GPT-5:")
print(f"• Utilisez des images de haute qualité")
print(f"• Posez des questions spécifiques")
print(f"• Exploitez le contexte éducatif")
print(f"• Combinez analyse textuelle et visuelle")
print(f"• Adaptez le niveau de complexité")


🖼️  EXEMPLES D'ANALYSE GPT-5

1. 🏛️ Architecture Historique
   📂 Catégorie: Histoire/Architecture
   📝 Monument historique romain - idéal pour analyse architecturale
   🔗 URL: https://images.unsplash.com/photo-1539037116277-4db20889f2d4...

2. 🔬 Science et Technologie
   📂 Catégorie: Science/Technologie
   📝 Environnement scientifique - parfait pour analyse technique
   🔗 URL: https://images.unsplash.com/photo-1532094349884-543bc11b234d...

3. 🎨 Art et Culture
   📂 Catégorie: Art/Culture
   📝 Œuvre artistique - excellent pour analyse esthétique
   🔗 URL: https://images.unsplash.com/photo-1541961017774-22349e4a1262...

4. 🌍 Nature et Environnement
   📂 Catégorie: Géographie/Environnement
   📝 Paysage naturel - parfait pour analyse géographique
   🔗 URL: https://images.unsplash.com/photo-1506905925346-21bda4d32df4...

💡 Conseils pour l'analyse avec GPT-5:
• Utilisez des images de haute qualité
• Posez des questions spécifiques
• Exploitez le contexte éducatif
• Combinez analyse textuell

### Тест: анализ демонстрационного изображения

Протестируем функцию анализа на конкретном изображении. GPT-5 опишет содержимое, определит визуальные элементы и предоставит структурированный анализ.

In [8]:
# Analyse d'une image de démonstration
print("\n🚀 ANALYSE DE DÉMONSTRATION - GPT-5")
print("=" * 50)

# Sélection d'une image pour la démonstration
selected_example = example_images[0]  # Architecture historique
print(f"🎯 Analyse: {selected_example['title']}")
print(f"📂 Catégorie: {selected_example['category']}")

# Test de l'analyse avec les différents modes
analysis_results = []

for mode in ['quick', 'detailed', 'educational']:
    print(f"\n🔍 Mode d'analyse: {mode.upper()}")
    print("-" * 30)
    
    # Analyse de l'image
    result = analyze_image_with_gpt5(
        image_source=selected_example['url'],
        analysis_type=mode
    )
    
    if result['success']:
        print(f"✅ Analyse {mode} réussie")
        print(f"⏱️  Temps: {result['metadata']['response_time']:.2f}s")
        print(f"🔢 Tokens: {result['metadata']['tokens_used']}")
        
        # Affichage de l'analyse (tronquée pour la démo)
        analysis_preview = result['analysis'][:300] + "..." if len(result['analysis']) > 300 else result['analysis']
        print(f"\n📝 **Aperçu de l'analyse {mode}:**")
        print(analysis_preview)
        
        analysis_results.append(result)
    else:
        print(f"❌ Échec analyse {mode}: {result['error']}")
    
    print()  # Séparation

# Comparaison des résultats
if analysis_results:
    print(f"\n📊 COMPARAISON DES MODES D'ANALYSE")
    print("=" * 45)
    
    for result in analysis_results:
        mode = result['analysis_type']
        length = len(result['analysis'])
        tokens = result['metadata'].get('tokens_used', {}).get('total_tokens', 'N/A')
        time = result['metadata']['response_time']
        
        print(f"{mode.capitalize():12} | {length:4d} chars | {tokens:>6} tokens | {time:5.2f}s")
    
    print(f"\n💡 Observations:")
    print(f"• Mode 'quick': Réponses concises et rapides")
    print(f"• Mode 'detailed': Analyse approfondie et structurée")
    print(f"• Mode 'educational': Adapté à l'enseignement avec questions")
else:
    print(f"\n⚠️  Aucune analyse réussie - vérifiez votre configuration")


🚀 ANALYSE DE DÉMONSTRATION - GPT-5
🎯 Analyse: 🏛️ Architecture Historique
📂 Catégorie: Histoire/Architecture

🔍 Mode d'analyse: QUICK
------------------------------

🔍 Analyse en cours...
📝 Type: quick
🖼️  Source: https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800


✅ Analyse quick réussie
⏱️  Temps: 7.76s
🔢 Tokens: {'prompt_tokens': 653, 'completion_tokens': 289, 'total_tokens': 942, 'cost': 0, 'is_byok': True, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.00370625, 'upstream_inference_prompt_cost': 0.00081625, 'upstream_inference_completions_cost': 0.00289}, 'completion_tokens_details': {'reasoning_tokens': 192, 'image_tokens': 0, 'audio_tokens': 0}}

📝 **Aperçu de l'analyse quick:**
Vue aérienne de la Gran Vía à Madrid au coucher du soleil. Au premier plan, l’édifice Metropolis, les façades éclairées et la circulation animent l’avenue sous un ciel orangé avec les montagnes à l’horizon.


🔍 Mode d'analyse: DETAILED
------------------------------

🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800


✅ Analyse detailed réussie
⏱️  Temps: 19.14s
🔢 Tokens: {'prompt_tokens': 758, 'completion_tokens': 1033, 'total_tokens': 1791, 'cost': 0, 'is_byok': True, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.0112775, 'upstream_inference_prompt_cost': 0.0009475, 'upstream_inference_completions_cost': 0.01033}, 'completion_tokens_details': {'reasoning_tokens': 320, 'image_tokens': 0, 'audio_tokens': 0}}

📝 **Aperçu de l'analyse detailed:**
1) Description générale
- Vue aérienne/haute d’un centre-ville dense au crépuscule.
- Une grande artère traverse l’image en diagonale, animée par des voitures et des passants.
- Au premier plan gauche, un immeuble d’angle à dôme sombre et ornementation riche; en face, d’autres façades néoclassiques ...


🔍 Mode d'analyse: EDUCATIONAL
------------------------------

🔍 Analyse en cours...
📝 Type: educational
🖼️  Source: https://images.unsplash.com/photo

✅ Analyse educational réussie
⏱️  Temps: 25.73s
🔢 Tokens: {'prompt_tokens': 804, 'completion_tokens': 1452, 'total_tokens': 2256, 'cost': 0, 'is_byok': True, 'prompt_tokens_details': {'cached_tokens': 0, 'cache_write_tokens': 0, 'audio_tokens': 0, 'video_tokens': 0}, 'cost_details': {'upstream_inference_cost': 0.015525, 'upstream_inference_prompt_cost': 0.001005, 'upstream_inference_completions_cost': 0.01452}, 'completion_tokens_details': {'reasoning_tokens': 704, 'image_tokens': 0, 'audio_tokens': 0}}

📝 **Aperçu de l'analyse educational:**
🎯 ANALYSE PÉDAGOGIQUE

1) Description accessible
- Vue panoramique d’un centre-ville européen au crépuscule. Une large avenue traverse l’image entre des immeubles historiques éclairés. Au premier plan, un bâtiment d’angle coiffé d’un dôme orné domine l’intersection; la ville s’étend jusqu’aux montag...


📊 COMPARAISON DES MODES D'ANALYSE
Quick        |  206 chars |    942 tokens |  7.76s
Detailed     | 2606 chars |   1791 tokens | 19.14s
Education

### Упражнение 1 : Создание персонализированного промпта для анализа

Три режима анализа (quick, detailed, educational) покрывают общие варианты использования, но реальные случаи часто требуют специализированных промптов. Цель — создать промпт, оптимизированный для конкретной области.

**Контекст** : Вы преподаватель и хотите использовать GPT-5 для анализа изображений в классе. Вы должны создать промпт, который формирует структурированный анализ, адаптированный к вашей дисциплине.

**Этапы** :
1. Выберите область (История, Науки, География, Искусство или Медицина)
2. Изучите существующие промпты в `analyze_image_with_gpt5()`, чтобы понять структуру
3. Создайте собственный промпт, который включает: техническое описание, специфическую для области лексику и 3 направляющих вопроса
4. Протестируйте ваш промпт на демонстрационном изображении и оцените качество ответа

**Подсказка** : Хороший промпт для анализа определяет роль ("Ты эксперт-преподаватель по..."), ожидаемый формат вывода (разделы, пункты) и уровень детализации. Ориентируйтесь на режим "educational".

In [9]:
# Exercice 1 : Creation d'un prompt d'analyse personnalise

# Etape 1 : Choix du domaine
mon_domaine = "..."  # TODO etudiant : "Histoire", "Sciences", "Geographie", "Art", ou "Medecine"

# Etape 2 : Creation du prompt personnalise
# Indice : incluez le role, le format de sortie, le niveau d'education, et 3 questions guides
mon_prompt_analyse = f"""..."""  # TODO etudiant : redigez votre prompt
# Structure suggeree :
# 1. Role ("Tu es un professeur expert en {mon_domaine}")
# 2. Format de sortie (sections avec titres)
# 3. Elements specifiques a analyser
# 4. Questions guides pour les etudiants

# Etape 3 : Test sur l'image de demonstration
image_test = example_images[0]['url']  # Architecture historique
# Indice : utilisez analyze_image_with_gpt5(image_test, prompt=mon_prompt_analyse)

# Etape 4 : Evaluation de la qualite
# TODO etudiant : la reponse contient-elle bien les sections demandees ?
# TODO etudiant : le vocabulaire est-il adapte au domaine ?
# TODO etudiant : les questions guides sont-elles pertinentes ?

print("Exercice a completer")

Exercice a completer


### Практические применения: доступность и педагогика

GPT-5 отлично справляется с двумя конкретными сценариями использования:
- **Alt text**: автоматическая генерация описаний доступности для веб-изображений
- **Педагогический анализ**: использование мультимодального зрения для создания образовательного контента на основе изображений

In [10]:
# Génération de descriptions d'accessibilité (Alt text)
if generate_alt_text and analysis_results:
    print("\n♿ GÉNÉRATION DE DESCRIPTIONS D'ACCESSIBILITÉ")
    print("=" * 55)
    
    # Prompt spécialisé pour l'accessibilité
    accessibility_prompt = f"""Génère une description d'accessibilité (alt text) pour cette image en {language}.
    
Critères :
- Maximum 125 caractères
- Description factuelle et objective
- Inclut les éléments essentiels pour la compréhension
- Évite les interprétations subjectives
- Adapté aux lecteurs d'écran

Format : Fournis UNIQUEMENT la description, sans formatage supplémentaire."""
    
    # Génération de l'alt text
    alt_result = analyze_image_with_gpt5(
        image_source=selected_example['url'],
        prompt=accessibility_prompt
    )
    
    if alt_result['success']:
        alt_text = alt_result['analysis'].strip()
        
        print(f"✅ Description d'accessibilité générée")
        print(f"📝 Alt text ({len(alt_text)} caractères) :")
        print(f'   "{alt_text}"')
        
        if len(alt_text) > 125:
            print(f"⚠️  Longueur dépassée ({len(alt_text)}/125 chars) - considérez une version plus courte")
        else:
            print(f"✅ Longueur optimale ({len(alt_text)}/125 chars)")
    else:
        print(f"❌ Erreur génération alt text: {alt_result['error']}")
else:
    print(f"\n⏭️  Génération alt text désactivée ou pas d'analyse disponible")


♿ GÉNÉRATION DE DESCRIPTIONS D'ACCESSIBILITÉ

🔍 Analyse en cours...
📝 Type: detailed
🖼️  Source: https://images.unsplash.com/photo-1539037116277-4db20889f2d4?w=800


✅ Description d'accessibilité générée
📝 Alt text (114 caractères) :
   "Vue aérienne de la Gran Vía à Madrid au coucher du soleil, dôme du Metropolis, immeubles illuminés et circulation."
✅ Longueur optimale (114/125 chars)


### Упражнение 2 : Генерация многоуровневых описаний доступности

Веб-доступность требует альтернативных текстов (alt text), адаптированных к контексту. Одно и то же изображение может требовать описаний разной длины в зависимости от использования: быстрая навигация (125 символов), подробное описание (для программ чтения с экрана) или педагогическое резюме.

**Цель** : Для изображения по вашему выбору сгенерировать три уровня описаний доступности с помощью GPT-5 и оценить их качество.

**Этапы** :
1. Выберите изображение из примеров или предоставьте собственный URL
2. Сгенерируйте короткий alt text (<= 125 символов) для быстрой навигации
3. Сгенерируйте подробное описание (200-400 символов) для программ чтения с экрана
4. Сгенерируйте педагогическое резюме, адаптированное к уровню "secondary"
5. Проверьте, что каждое описание является фактическим и объективным

**Подсказка** : Составьте специальные промпты для каждого уровня описания. Явно укажите максимальную длину и контекст использования в промпте.

In [11]:
# Exercice 2 : Generation de descriptions d'accessibilite multi-niveaux

# Etape 1 : Choix de l'image
mon_image = "https://images.unsplash.com/photo-1506905925346-21bda4d32df4?w=800"  # Paysage naturel
# TODO etudiant : remplacez par l'URL de votre choix si vous le souhaitez

# Etape 2 : Prompt pour alt text court (<= 125 caracteres)
prompt_court = "..."  # TODO etudiant : prompt demandant une description en <= 125 caracteres
# Indice : specifiez "factual, objective, no interpretation, max 125 characters"

# Etape 3 : Prompt pour description detaillee (200-400 caracteres)
prompt_detaille = "..."  # TODO etudiant : prompt pour lecteurs d'ecran
# Indice : incluez elements essentiels, couleurs, disposition, actions visibles

# Etape 4 : Prompt pour resume pedagogique niveau "secondary"
prompt_pedago = "..."  # TODO etudiant : prompt adapte a des eleves du secondaire
# Indice : utilisez un vocabulaire accessible, posez des questions de reflexion

# Generation (decommentez pour tester)
# resultats = {}
# for nom, prompt in [("court", prompt_court), ("detaille", prompt_detaille), ("pedago", prompt_pedago)]:
#     r = analyze_image_with_gpt5(mon_image, prompt=prompt)
#     if r['success']:
#         resultats[nom] = r['analysis']
#         print(f"\n--- {nom.upper()} ({len(r['analysis'])} chars) ---")
#         print(r['analysis'][:300])

# Etape 5 : Verification
# TODO etudiant : verifiez que le texte court fait bien <= 125 caracteres
# TODO etudiant : verifiez que les descriptions sont factuelles (pas d'interpretation subjective)

print("Exercice a completer")

Exercice a completer


Демонстрационные анализы завершены. Интерактивный режим ниже позволяет экспериментировать с вашими собственными изображениями и персонализированными подсказками для анализа.

In [12]:
# Mode interactif - Analyse d'image personnalisée
if notebook_mode == "interactive" and not skip_widgets:
    print("\n🎨 MODE INTERACTIF - ANALYSE PERSONNALISÉE")
    print("=" * 55)
    
    print("\n💡 Analysez votre propre image avec GPT-5:")
    print("Formats supportés: URL https:// ou chemin local")
    print("(Laissez vide pour passer à la suite)")
    
    try:
        user_image = input("\n🖼️  URL ou chemin de votre image: ").strip()

        if user_image:
            # Paramètres d'analyse personnalisés
            print("\n⚙️  Paramètres d'analyse (appuyez Entrée pour défaut):")
            custom_mode = input(f"📊 Mode [{analysis_mode}]: ").strip() or analysis_mode
            custom_prompt = input("📝 Prompt personnalisé (optionnel): ").strip()

            print(f"\n🔍 Analyse de votre image en cours...")

            # Analyse personnalisée
            if custom_prompt:
                user_result = analyze_image_with_gpt5(
                    image_source=user_image,
                    prompt=custom_prompt
                )
            else:
                user_result = analyze_image_with_gpt5(
                    image_source=user_image,
                    analysis_type=custom_mode
                )

            if user_result['success']:
                print(f"\n🎉 Analyse réussie!")
                print(f"⏱️  Temps: {user_result['metadata']['response_time']:.2f}s")
                print(f"\n📝 **Analyse GPT-5:**")
                print(user_result['analysis'])

                # Option de sauvegarde
                if export_analysis:
                    try:
                        save_choice = input("\n💾 Sauvegarder cette analyse ? (o/N): ").strip().lower()
                        if save_choice in ['o', 'oui', 'y', 'yes']:
                            output_dir = current_path / 'outputs' / 'gpt5_analysis'
                            output_dir.mkdir(parents=True, exist_ok=True)

                            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                            analysis_file = output_dir / f"gpt5_analysis_{timestamp}.json"

                            with open(analysis_file, 'w', encoding='utf-8') as f:
                                json.dump(user_result, f, indent=2, ensure_ascii=False)

                            print(f"💾 Analyse sauvegardée: {analysis_file}")
                    except Exception as save_err:
                        if "StdinNotImplemented" in type(save_err).__name__:
                            print("\n⏭️  Sauvegarde automatique ignorée en mode batch")
                        else:
                            raise
            else:
                print(f"\n❌ Erreur: {user_result['error']}")
                print(f"🔍 Source: {user_result['image_source']}")
        else:
            print("\n⏭️  Mode interactif ignoré")

    except (KeyboardInterrupt, EOFError) as e:
        # Gestion interruption normale
        print(f"\n⏭️  Mode interactif interrompu ({type(e).__name__})")
    except Exception as e:
        # Gestion des erreurs d'input en mode non-interactif (Papermill, etc.)
        error_type = type(e).__name__
        if "StdinNotImplemented" in error_type or "input" in str(e).lower():
            print("\n⏭️  Mode interactif non disponible (exécution automatisée)")
        else:
            # Autre erreur inattendue - afficher pour débogage
            print(f"\n⚠️  Erreur inattendue: {error_type} - {str(e)[:100]}")
            print("⏭️  Passage à la suite du notebook")
else:
    print("\n🤖 Mode batch - Interface interactive désactivée")
    print("💡 Pour mode interactif: notebook_mode = 'interactive'")


🎨 MODE INTERACTIF - ANALYSE PERSONNALISÉE

💡 Analysez votre propre image avec GPT-5:
Formats supportés: URL https:// ou chemin local
(Laissez vide pour passer à la suite)

⏭️  Mode interactif non disponible (exécution automatisée)


После рассмотрения интерактивного режима следующая ячейка представляет основные прикладные сценарии использования визуального анализа с помощью GPT-5 в профессиональном и педагогическом контексте.

In [13]:
# Cas d'usage pédagogiques avec GPT-5
print("\n🎓 CAS D'USAGE PÉDAGOGIQUES GPT-5")
print("=" * 45)

# Exemples d'applications éducatives
educational_use_cases = {
    "Histoire": {
        "description": "Analyse de documents historiques, œuvres d'art, monuments",
        "exemple": "Analyser une fresque renaissance pour comprendre le contexte social",
        "prompt_template": "Analyse cette image historique pour des étudiants en histoire. Explique le contexte historique, les éléments symboliques, et l'importance culturelle."
    },
    
    "Sciences": {
        "description": "Analyse d'expériences, schémas scientifiques, phénomènes naturels",
        "exemple": "Expliquer un diagramme de cellule ou une réaction chimique",
        "prompt_template": "En tant qu'enseignant de sciences, explique cette image scientifique. Identifie les éléments techniques et leur fonctionnement."
    },
    
    "Géographie": {
        "description": "Étude de paysages, cartes, phénomènes géologiques",
        "exemple": "Analyser une photo satellite ou un paysage géographique",
        "prompt_template": "Analyse cette image géographique pour des étudiants. Explique les formations géologiques, le climat, et l'impact humain visible."
    },
    
    "Art et Culture": {
        "description": "Critique artistique, analyse stylistique, histoire de l'art",
        "exemple": "Comprendre les techniques d'un tableau impressionniste",
        "prompt_template": "Fais une analyse artistique de cette œuvre pour des étudiants en art. Inclus style, techniques, et signification culturelle."
    },
    
    "Médecine": {
        "description": "Analyse d'imagerie médicale, anatomie, cas cliniques",
        "exemple": "Expliquer une radiographie ou un schéma anatomique",
        "prompt_template": "Analyse cette image médicale pour des étudiants en médecine. Explique l'anatomie visible et les points d'intérêt clinique."
    }
}

# Affichage des cas d'usage
for domain, info in educational_use_cases.items():
    print(f"\n📚 **{domain}**")
    print(f"   📋 {info['description']}")
    print(f"   💡 Exemple: {info['exemple']}")
    print(f"   📝 Template de prompt disponible")

print(f"\n🎯 Avantages GPT-5 pour l'éducation:")
print(f"• **Multimodal** : Analyse texte + image simultanée")
print(f"• **Contextuel** : Comprend le niveau éducatif")
print(f"• **Adaptable** : Ajuste le vocabulaire selon l'audience")
print(f"• **Interactif** : Permet les questions de suivi")
print(f"• **Précis** : Analyse détaillée et factuelle")

print(f"\n📝 Exemple de workflow pédagogique:")
print(f"1. Sélection d'image pertinente au cours")
print(f"2. Analyse GPT-5 avec prompt éducatif")
print(f"3. Génération de questions pour les étudiants")
print(f"4. Discussion interactive basée sur l'analyse")
print(f"5. Évaluation de la compréhension")


🎓 CAS D'USAGE PÉDAGOGIQUES GPT-5

📚 **Histoire**
   📋 Analyse de documents historiques, œuvres d'art, monuments
   💡 Exemple: Analyser une fresque renaissance pour comprendre le contexte social
   📝 Template de prompt disponible

📚 **Sciences**
   📋 Analyse d'expériences, schémas scientifiques, phénomènes naturels
   💡 Exemple: Expliquer un diagramme de cellule ou une réaction chimique
   📝 Template de prompt disponible

📚 **Géographie**
   📋 Étude de paysages, cartes, phénomènes géologiques
   💡 Exemple: Analyser une photo satellite ou un paysage géographique
   📝 Template de prompt disponible

📚 **Art et Culture**
   📋 Critique artistique, analyse stylistique, histoire de l'art
   💡 Exemple: Comprendre les techniques d'un tableau impressionniste
   📝 Template de prompt disponible

📚 **Médecine**
   📋 Analyse d'imagerie médicale, anatomie, cas cliniques
   💡 Exemple: Expliquer une radiographie ou un schéma anatomique
   📝 Template de prompt disponible

🎯 Avantages GPT-5 pour l'éducat

## 🎯 Резюме и лучшие практики

### ✅ Что вы изучили

- [ ] **Конфигурация GPT-5** : OpenRouter API и оптимальные параметры
- [ ] **Мультимодальный анализ** : Сочетание текста + изображения
- [ ] **Образовательные промпты** : Адаптация к уровню обучения
- [ ] **Педагогические сценарии использования** : Практические применения по областям
- [ ] **Оптимизация** : Параметры качества и производительности

### 🚀 Следующие шаги

1. **Экспериментируйте** со своими собственными образовательными изображениями
2. **Тестируйте** разные уровни анализа в зависимости от вашей аудитории
3. **Интегрируйте** GPT-5 в ваши педагогические workflows
4. **Комбинируйте** с DALL-E 3 для генерации + анализа
5. **Изучайте** продвинутые notebooks (Module 02)

### 💡 Советы по использованию

**✅ Лучшие практики:**
- Используйте изображения высокого качества
- Адаптируйте prompt к образовательному уровню
- Сочетайте визуальный и контекстный анализ
- Генерируйте педагогические вопросы
- Сохраняйте успешные анализы

**❌ Избегайте:**
- Слишком маленьких или некачественных изображений
- Расплывчатых или общих prompts
- Забывать об образовательном контексте
- Игнорировать ограничения модели
- Не проверять фактическую точность

### 🔗 Дополнительные ресурсы

- **Документация OpenRouter** : [openrouter.ai](https://openrouter.ai)
- **Руководство GPT-5** : Мультимодальные возможности
- **Образовательные шаблоны** : `docs/genai-phase2-templates.md`
- **Стандарты CoursIA** : `docs/genai-images-development-standards.md`

### 📖 Научные ссылки

- **OpenAI** (2025). *GPT-5 System Card & Model Page*. — Мультимодальная модель (текст + зрение), окно контекста **400 000 tokens**, максимум 128 000 tokens на выходе. Официальные источники : openai.com/gpt-5/ и developers.openai.com/api/docs/models/gpt-5.
- **OpenAI** (2024). *GPT-4o System Card*. — Мультимодальный предшественник GPT-5 (объединённые зрение + текст), основа мультимодального API, используемого через OpenRouter в этом notebook.

### Упражнение 3: Синтез — полный workflow мультимодального анализа

**Оценочная продолжительность:** 15-20 минут

### Цель

Создать персонализированный workflow анализа изображений с GPT-5 для конкретного педагогического сценария использования (история, науки, география, искусство или медицина).

### Инструкции

1. **Выберите область** из 5 педагогических сценариев использования, представленных в notebook
2. **Выберите изображение**, релевантное для этой области (публичный URL или локальное изображение)
3. **Создайте персонализированный prompt**, адаптированный к целевому образовательному уровню
4. **Реализуйте функцию анализа**, которая сочетает 3 режима (quick, detailed, educational)
5. **Сгенерируйте педагогические вопросы**, которые GPT-5 предлагает для студентов

### Подсказки

- Подсказка 1: Используйте функцию `analyze_image_with_gpt5()` с параметром `prompt` для вашего персонализированного prompt
- Подсказка 2: Режим "educational" автоматически генерирует 3 вопроса для размышления для студентов
- Подсказка 3: Адаптируйте лексику prompt к уровню (elementary, secondary, university)

In [14]:
# TODO: Implémentez l'analyse personnalisée pour votre domaine

# 1. Sélection du domaine et configuration
domain = "..."  # "Histoire", "Sciences", "Géographie", "Art", "Médecine"
niveau = "university"  # Niveau éducatif cible

# 2. Image à analyser (choisir une URL ou chemin local)
image_source = "https://images.unsplash.com/photo-..."  # Remplacer par votre image

# 3. Création du prompt personnalisé
# Indice: Inspirez-vous des templates dans educational_use_cases
custom_prompt = f"""Tu es un professeur expert en {domain}.
Analyse cette image pour des étudiants de niveau {niveau}.
Inclus :
1. Description accessible du sujet
2. Éléments techniques importants à observer
3. Contexte historique ou scientifique
4. 3 questions de réflexion pour les étudiants
"""

# 4. Implémenter l'analyse avec les 3 modes
# Indice: Utilisez analyze_image_with_gpt5() avec analysis_type et prompt
def analyse_complete(image_url, prompt, domain):
    """
    Analyse une image avec les 3 modes et retourne les résultats.
    
    Args:
        image_url: URL ou chemin de l'image
        prompt: Prompt personnalisé pour le mode educational
        domain: Domaine d'étude pour le contexte
    
    Returns:
        Dict avec les 3 analyses (quick, detailed, educational)
    """
    results = {}
    
    # TODO: Implémentez les 3 analyses
    # - Mode quick pour une vue d'ensemble
    # - Mode detailed pour l'analyse technique
    # - Mode educational avec votre prompt personnalisé
    
    pass  # Remplacer par votre implémentation
    
    return results

# 5. Exécuter l'analyse et afficher les résultats
# Indice: Utilisez pprint.json pour un affichage lisible des résultats
pass

### Critères de succès

- [ ] Le prompt personnalisé est adapté au domaine et au niveau éducatif
- [ ] Les 3 modes d'analyse sont implémentés (quick, detailed, educational)
- [ ] Les questions pédagogiques sont générées par le mode educational
- [ ] Les résultats sont structurés et présentés clairement
- [ ] Le temps de réponse total est inférieur à 60 secondes pour les 3 analyses

### Extension (optionnelle)

Pour les étudiants avancés :
- Ajouter une fonction de **comparaison** entre 2 images du même domaine
- Implémenter une **évaluation** des réponses étudiant avec GPT-5
- Créer un **quiz interactif** basé sur les questions générées

## Conclusion

GPT-5 image generation (via OpenRouter) incarne l'approche **multimodale native** : le modèle ne se contente pas de générer des images à partir d'un prompt texte, il analyse et produit dans un même flux, ce qui ouvre trois modes d'usage pédagogique distincts — quick (206 chars), detailed (2606 chars), educational (2908 chars). Le ratio coût d'inférence / richesse du contenu est directement exposé à l'utilisateur par le choix du mode.

**Apport spécifique de la multimodalité native** : la génération automatique d'alt text (114 chars optimal, sous le seuil 125 chars des référentiels d'accessibilité WCAG) intègre la description d'image comme **livrable de première classe**, pas comme un post-traitement optionnel. Pour les 5 cas d'usage éducatif couverts (Histoire, Sciences, Géographie, Art et Culture, Médecine), c'est un gain d'accessibilité substantiel par rapport aux modèles purement text-to-image.

**Positionnement dans l'écosystème** : GPT-5 complète la palette du cours en couvrant l'axe *compréhension + génération*, là où DALL-E 3 se concentre sur la génération seule et où SD XL Turbo expose les rouages internes. Les 26 modèles GPT-5 détectés via l'API OpenRouter ouvrent un terrain d'expérimentation comparative large mais introduisent une dépendance forte au fournisseur de route.

**Pour aller plus loin** : explorer le notebook `04-1-Educational-Content-Generation` qui orchestre plusieurs modèles pour produire un livrable pédagogique complet (image + alt text + métadonnées structurées).
